In [ ]:
import duckdb
import pandas

duck = duckdb.connect("Абрамов_Olist.duckdb")
duck.execute("DROP SCHEMA IF EXISTS olist CASCADE;")
duck.execute("CREATE SCHEMA olist;")

В качестве идентификаторов, в датасете используются HEX-строки. Для удобства анализа, подготовим функцию, которая преобразует их в числа.

In [2]:
import hashlib

def hash_id(hex_id: str) -> int:
    hash_obj = hashlib.sha3_384(hex_id.encode()).digest()
    return int.from_bytes(hash_obj[:4], 'little', signed = True) # 4 байта = 32 бита = integer
    # Это может привести к коллизиям (два hex_id преобразуются в одинаковый целочисленный id),
    # но будем надеяться, что такого не произойдет. В крайнем случае, это будет очень легко заметить.

hash_id("3442f8959a84dea7ee197c632cb2df15")

-1881377438

### olist_geolocation_dataset.csv

In [3]:
geolocation = pandas.read_csv("./Olist/olist_geolocation_dataset.csv")
display(geolocation)
geolocation.nunique()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP
...,...,...,...,...,...
1000158,99950,-28.068639,-52.010705,tapejara,RS
1000159,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,99950,-28.071855,-52.014716,tapejara,RS
1000161,99980,-28.388932,-51.846871,david canabarro,RS


geolocation_zip_code_prefix     19015
geolocation_lat                717360
geolocation_lng                717613
geolocation_city                 8011
geolocation_state                  27
dtype: int64

В качестве первичного ключа наверняка подойдёт пара `(lat, lng)`. Тем не менее в других таблицах эти значения не встречаются, а связи построены неявно. Например, в таблицах `customers` и `sellers` присутствуют три столбца: `zip_code_prefix`, `city`, `state`. При этом нет гарантии, что эта тройка встречается в таблице `geolocation` (более того, есть обратные примеры). Провести явное объединение данных продавцов и покупателей с геолокациями не представляется возможным.

Учитывая отсутствие необходимости обработки `lat` и `lng` в запросах, а также невозможность выбрать явного кандидата для первичного ключа, считаю целесообразным вовсе не использовать таблицу `geolocation` в анализе, а столбцы `city` и `state` оставить в таблицах продавцов и покупателей.

Более того, кажется необходимым ввести предобработку значений `city`: в данных есть примеры, отличающиеся лишь написанием: где-то используются портгульаские буквы (ã, ç и др.), а где-то написание полностью на английском. Подготовим словарь для приведения всех строк к английскому написанию: бразильские буквы точно не будут важны при дальнейшем анализе.

In [4]:
letters_to_replace = {
    'ã': 'a',
    'á': 'a',
    'â': 'a',
    'ú': 'u',
    'ç': 'c',
    'í': 'i',
    "ô": "o",
    "ó": "o",
    "õ": "o",
    "é": "e",
    "ê": "e",
    "ü": "u",
    # Уберем апострофы и дефисы
    "'": "", 
    "-": "",
}

### olist_sellers_dataset.csv

In [5]:
sellers = pandas.read_csv("./Olist/olist_sellers_dataset.csv")
sellers

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
...,...,...,...,...
3090,98dddbc4601dd4443ca174359b237166,87111,sarandi,PR
3091,f8201cab383e484733266d1906e2fdfa,88137,palhoca,SC
3092,74871d19219c7d518d0090283e03c137,4650,sao paulo,SP
3093,e603cf3fec55f8697c9059638d6c8eb5,96080,pelotas,RS


Как описано ранее, столбцы `zip_code_prefix`, `city` и `state` оставим в этой таблице. В столбце `city` заменим португальские буквы на английские. Столбец `seller_id` преобразуем в численный тип.

In [6]:
sellers["id"] = sellers["seller_id"].apply(hash_id)
for to_replace, value in letters_to_replace.items():
    sellers["seller_city"] = sellers["seller_city"].str.replace(to_replace, value)
sellers = sellers[["id", "seller_zip_code_prefix", "seller_city", "seller_state"]]
sellers

,id,seller_zip_code_prefix,seller_city,seller_state
0,-1881377438,13023,campinas,SP
1,-1807436828,13844,mogi guacu,SP
2,-2141338086,20031,rio de janeiro,RJ
3,-572600908,4195,sao paulo,SP
4,-333364572,12914,braganca paulista,SP
...,...,...,...,...
3090,610029896,87111,sarandi,PR
3091,386540530,88137,palhoca,SC
3092,-738389240,4650,sao paulo,SP
3093,-1284387824,96080,pelotas,RS


In [7]:
duck.execute("""
CREATE TABLE olist.sellers (
    id INTEGER PRIMARY KEY,
    zip_code_prefix INTEGER NOT NULL,
    city VARCHAR NOT NULL,
    state VARCHAR NOT NULL
);
""")
duck.execute("INSERT INTO olist.sellers SELECT * FROM sellers")

### olist_customers_dataset.csv

In [8]:
customers = pandas.read_csv("./Olist/olist_customers_dataset.csv")
customers

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
...,...,...,...,...,...
99436,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,3937,sao paulo,SP
99437,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,6764,taboao da serra,SP
99438,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115,fortaleza,CE
99439,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS


Таблицу покупателей преобразуем аналогично продавцам. Единственное отличие - таблица покупателей содержит два идентификатора. В соответствии с описанием данных, для каждого заказа создаётся новая запись с уникальным `customer_id`, а для выявления возвращающихся покупателей используется `customer_unique_id`

In [9]:
customers["customer_id"] = customers["customer_id"].apply(hash_id)
customers["customer_unique_id"] = customers["customer_unique_id"].apply(hash_id)
for to_replace, value in letters_to_replace.items():
    customers["customer_city"] = customers["customer_city"].str.replace(to_replace, value)
customers

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,-1030598386,1290849265,14409,franca,SP
1,-653578874,-1541764783,9790,sao bernardo do campo,SP
2,1839707216,1201662163,1151,sao paulo,SP
3,-640794803,1487701330,8775,mogi das cruzes,SP
4,2110311777,-1634706095,13056,campinas,SP
...,...,...,...,...,...
99436,867097572,-1185791115,3937,sao paulo,SP
99437,-2002978370,-1476341675,6764,taboao da serra,SP
99438,-1541758221,-323049518,60115,fortaleza,CE
99439,864641411,-2108416896,92120,canoas,RS


In [10]:
duck.execute("""
CREATE TABLE olist.customers (
    id INTEGER PRIMARY KEY,
    unique_id INTEGER NOT NULL,
    zip_code_prefix INTEGER NOT NULL,
    city VARCHAR NOT NULL,
    state VARCHAR NOT NULL
);
""")
# Наверняка мы захотим искать по unique_id, поэтому сразу подгоготовим индекс
duck.execute("CREATE INDEX customer_unique_id ON olist.customers (unique_id)")

duck.execute("INSERT INTO olist.customers SELECT * FROM customers")

### olist_orders_dataset.csv

In [11]:
orders = pandas.read_csv("./Olist/olist_orders_dataset.csv")
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00


Здесь ничего примечательного: преобразуем идентификаторы в численный формат, а остальные столбцы оставим без изменений. При формировании таблицы дополнительно преобразуем даты в соответствующий формат, а также не забудем про внешний ключ с таблицей `customers` по столбцу `customer_id`.

In [12]:
orders["order_id"] = orders["order_id"].apply(hash_id)
orders["customer_id"] = orders["customer_id"].apply(hash_id)
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,1317765622,229024233,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,1324528439,126778905,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,348336675,-1952515779,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,308376798,-1765145932,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,81703435,1221251903,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
...,...,...,...,...,...,...,...,...
99436,-1178475214,1393557541,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,-791826529,-863251369,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,827020936,-1847359780,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,-385911142,-864746792,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00


In [13]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [14]:
orders.isna().sum(axis = 0)

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [15]:
duck.execute("""
CREATE TYPE olist.order_status AS ENUM (
    'delivered',
    'shipped',
    'canceled',
    'unavailable',
    'invoiced',
    'processing',
    'created',
    'approved'
);
""")
duck.execute("""
CREATE TABLE olist.orders (
    id INTEGER PRIMARY KEY,
    customer_id INTEGER UNIQUE NOT NULL,
    status olist.order_status NOT NULL,
    purchase DATETIME NOT NULL,
    approved_at DATETIME,
    delivered_carrier DATETIME,
    delivered_customer DATETIME,
    estimated_delivery DATE NOT NULL,
             
    FOREIGN KEY (customer_id) REFERENCES olist.customers(id)
);
""")
duck.execute("INSERT INTO olist.orders SELECT * FROM orders")

### olist_order_reviews_dataset.csv

In [16]:
order_reviews = pandas.read_csv("./Olist/olist_order_reviews_dataset.csv")
order_reviews

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53
...,...,...,...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07 00:00:00,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09 00:00:00,2017-12-11 20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22 00:00:00,2018-03-23 09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01 00:00:00,2018-07-02 12:59:13


In [17]:
order_reviews["order_id"].value_counts()

order_id
c88b1d1b157a9999ce368f218a407141    3
8e17072ec97ce29f0e1f111e598b0c85    3
df56136b8031ecd28e200bb18e6ddb2e    3
03c939fd7fd3b38f8485a0f95798f1f6    3
5cb890a68b91b6158d69257e4e2bc359    2
                                   ..
5b4e9a12d219f34f5c2de9f8d620b19d    1
a6da096d974acc000962856d7386448a    1
75e0647c26de647eca3421e9cc66c9da    1
bad0467c52f23cdc71e9fa139d4a8afd    1
90531360ecb1eec2a1fbb265a0db0508    1
Name: count, Length: 98673, dtype: int64

In [18]:
order_reviews[order_reviews["order_id"] == "c88b1d1b157a9999ce368f218a407141"]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
1985,ffb8cff872a625632ac983eb1f88843c,c88b1d1b157a9999ce368f218a407141,3,NaN,NaN,2017-07-22 00:00:00,2017-07-26 13:41:07
82525,202b5f44d09cd3cfc0d6bd12f01b044c,c88b1d1b157a9999ce368f218a407141,5,NaN,NaN,2017-07-22 00:00:00,2017-07-26 13:40:22
89360,fb96ea2ef8cce1c888f4d45c8e22b793,c88b1d1b157a9999ce368f218a407141,5,NaN,NaN,2017-07-21 00:00:00,2017-07-26 13:45:15


In [19]:
order_reviews["review_id"].value_counts()

review_id
7b606b0d57b078384f0b58eac1d41d78    3
dbdf1ea31790c8ecfcc6750525661a9b    3
32415bbf6e341d5d517080a796f79b5c    3
0c76e7a547a531e7bf9f0b99cba071c1    3
4219a80ab469e3fc9901437b73da3f75    3
                                   ..
95e01591b0e69a2fab382b0c562d4e20    1
93611e0327d6a1769d1e68cf3caa242d    1
983c47de74278257f99c4b918fd380f1    1
ca475b77fcc618551ef9d516c3f61b88    1
efe49f1d6f951dd88b51e6ccd4cc548f    1
Name: count, Length: 98410, dtype: int64

In [20]:
order_reviews[order_reviews["review_id"] == "7b606b0d57b078384f0b58eac1d41d78"]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
7500,7b606b0d57b078384f0b58eac1d41d78,f3028a8f41ea1ee2b461420913663f97,5,NaN,NaN,2017-02-15 00:00:00,2017-02-21 23:30:22
59859,7b606b0d57b078384f0b58eac1d41d78,2deb17060fc1ce18a85eba953ddcdeaf,5,NaN,NaN,2017-02-15 00:00:00,2017-02-21 23:30:22
61069,7b606b0d57b078384f0b58eac1d41d78,2f8f31eb2f7b6572836d662a6625c8e4,5,NaN,NaN,2017-02-15 00:00:00,2017-02-21 23:30:22


Получается, у одного заказа может быть несколько отзывов, а один отзыв может относиться сразу к нескольким заказам (отношение многие-ко-многим). Таким образом, в таблице нет явного первичного ключа, в связи с чем введем его "искусственно" как сумму `review_id` + `order_id`. Также важно настроить внешний ключ по столбцу `order_id` на таблицу `orders` и корректно сохранить два столбца с датами.

In [21]:
order_reviews["review_id"] = order_reviews["review_id"].apply(hash_id)
order_reviews["order_id"] = order_reviews["order_id"].apply(hash_id)
order_reviews.insert(0, "id", ((order_reviews["review_id"] + order_reviews["order_id"]) / 2).astype(int))
order_reviews

,id,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,544254373,688575595,399933151,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,-550648117,196355703,-1297651937,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,-2026551319,-2069253212,-1983849426,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,1408411033,1975214664,841607403,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,-209442477,1119630394,-1538515349,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53
...,...,...,...,...,...,...,...,...
99219,-665834509,-743953346,-587715673,5,NaN,NaN,2018-07-07 00:00:00,2018-07-14 17:18:30
99220,-545676363,-545846540,-545506186,5,NaN,NaN,2017-12-09 00:00:00,2017-12-11 20:06:42
99221,-26242856,-541407550,488921837,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22 00:00:00,2018-03-23 09:10:43
99222,-893024145,-412170793,-1373877497,4,NaN,NaN,2018-07-01 00:00:00,2018-07-02 12:59:13


In [22]:
order_reviews["review_score"].value_counts()

review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64

In [23]:
order_reviews.isna().sum(axis = 0)

id                             0
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [24]:
duck.execute("""
CREATE TABLE olist.order_reviews (
    id INTEGER PRIMARY KEY,
    review_id INTEGER NOT NULL,
    order_id INTEGER NOT NULL,
    score INTEGER NOT NULL CHECK(score >= 1 AND score <= 5),
    title TEXT,
    message TEXT,
    created_at DATE NOT NULL,
    answered_at DATETIME NOT NULL,

    FOREIGN KEY (order_id) REFERENCES olist.orders(id)
);
""")
# Наверняка мы захотим искать по review_id и order_id, поэтому сразу подгоготовим индексы
duck.execute("CREATE INDEX order_reviews_review_id ON olist.order_reviews (review_id)")
duck.execute("CREATE INDEX order_reviews_order_id ON olist.order_reviews (order_id)")

duck.execute("INSERT INTO olist.order_reviews SELECT * FROM order_reviews")

### olist_order_payments_dataset.csv

In [25]:
order_payments = pandas.read_csv("./Olist/olist_order_payments_dataset.csv")
order_payments

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
...,...,...,...,...,...
103881,0406037ad97740d563a178ecc7a2075c,1,boleto,1,363.31
103882,7b905861d7c825891d6347454ea7863f,1,credit_card,2,96.80
103883,32609bbb3dd69b3c066a6860554a77bf,1,credit_card,1,47.77
103884,b8b61059626efa996a60be9bb9320e10,1,credit_card,5,369.54


In [26]:
order_payments["order_id"].value_counts()

order_id
fa65dad1b0e818e3ccc5cb0e39231352    29
ccf804e764ed5650cd8759557269dc13    26
285c2e15bebd4ac83635ccc563dc71f4    22
895ab968e7bb0d5659d16cd74cd1650c    21
fedcd9f7ccdc8cba3a18defedd1a5547    19
                                    ..
6d2a30c9b7dcee3ed507dc9a601f99e7     1
a7737f6d9208dd56ea498a322ed3c37f     1
646e62df54f3e236eb6d5ff3b31429b8     1
e115da7a49ec2acf622e1f31da65cfb9     1
28bbae6599b09d39ca406b747b6632b1     1
Name: count, Length: 99440, dtype: int64

In [27]:
order_payments[order_payments["order_id"] == 'fa65dad1b0e818e3ccc5cb0e39231352']

,order_id,payment_sequential,payment_type,payment_installments,payment_value
4885,fa65dad1b0e818e3ccc5cb0e39231352,27,voucher,1,66.02
9985,fa65dad1b0e818e3ccc5cb0e39231352,4,voucher,1,29.16
14321,fa65dad1b0e818e3ccc5cb0e39231352,1,voucher,1,3.71
17274,fa65dad1b0e818e3ccc5cb0e39231352,9,voucher,1,1.08
19565,fa65dad1b0e818e3ccc5cb0e39231352,10,voucher,1,12.86
23074,fa65dad1b0e818e3ccc5cb0e39231352,2,voucher,1,8.51
24879,fa65dad1b0e818e3ccc5cb0e39231352,25,voucher,1,3.68
28330,fa65dad1b0e818e3ccc5cb0e39231352,5,voucher,1,0.66
29648,fa65dad1b0e818e3ccc5cb0e39231352,6,voucher,1,5.02
32519,fa65dad1b0e818e3ccc5cb0e39231352,11,voucher,1,4.03


По одному заказу может быть несколько платежей. Порядок платежей указан в столбце `payment_sequential`. Таким образом, первичным ключом можно считать пару `(order_id, payment_sequential)`. 

In [28]:
order_payments["order_id"] = order_payments["order_id"].apply(hash_id)
order_payments.insert(0, "id", (order_payments["order_id"] + order_payments["payment_sequential"] - 1).astype(int))
order_payments = order_payments.drop(columns = ['payment_sequential'])
order_payments

,id,order_id,payment_type,payment_installments,payment_value
0,9114739,9114739,credit_card,8,99.33
1,1323334785,1323334785,credit_card,1,24.39
2,1340719277,1340719277,credit_card,1,65.71
3,-796092206,-796092206,credit_card,8,107.78
4,-1887229744,-1887229744,credit_card,2,128.45
...,...,...,...,...,...
103881,-1657895370,-1657895370,boleto,1,363.31
103882,134733137,134733137,credit_card,2,96.80
103883,1786764126,1786764126,credit_card,1,47.77
103884,408594452,408594452,credit_card,5,369.54


In [29]:
order_payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [30]:
duck.execute("""
CREATE TYPE olist.payment_type AS ENUM (
    'not_defined',
    'credit_card',
    'boleto',
    'voucher',
    'debit_card'
);
""")
duck.execute("""
CREATE TABLE olist.payments (
    id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    type olist.payment_type NOT NULL,
    installments SMALLINT NOT NULL,
    value DOUBLE NOT NULL,
             
    FOREIGN KEY (order_id) REFERENCES olist.orders(id)
);
""")
# Наверняка мы захотим искать по order_id, поэтому сразу подгоготовим индекс
duck.execute("CREATE INDEX payments_order_id ON olist.payments (order_id)")

duck.execute("INSERT INTO olist.payments SELECT * FROM order_payments")

### product_category_name_translation.csv

In [31]:
translation = pandas.read_csv("./Olist/product_category_name_translation.csv")
translation

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
...,...,...
66,flores,flowers
67,artes_e_artesanato,arts_and_craftmanship
68,fraldas_higiene,diapers_and_hygiene
69,fashion_roupa_infanto_juvenil,fashion_childrens_clothes


Эта таблица позволяет переводить названия товаров с португальского на английский. Хочется ожидать, что значения в столбце `product_category_name` уникальны, и их можно использовать в качестве первичного ключа. Тем не менее для удобства, добавим к каждой строке численный идентификатор, чтобы было удобнее объединять.

In [32]:
translation = translation.reset_index() # Добавляем столбец index с идентификатором строки
translation

,index,product_category_name,product_category_name_english
0,0,beleza_saude,health_beauty
1,1,informatica_acessorios,computers_accessories
2,2,automotivo,auto
3,3,cama_mesa_banho,bed_bath_table
4,4,moveis_decoracao,furniture_decor
...,...,...,...
66,66,flores,flowers
67,67,artes_e_artesanato,arts_and_craftmanship
68,68,fraldas_higiene,diapers_and_hygiene
69,69,fashion_roupa_infanto_juvenil,fashion_childrens_clothes


In [33]:
duck.execute("""
CREATE TABLE olist.translation (
    id INTEGER PRIMARY KEY,
    portuguese VARCHAR NOT NULL UNIQUE,
    english VARCHAR NOT NULL
);
""")
duck.execute("INSERT INTO olist.translation SELECT * FROM translation")

### olist_products_dataset.csv

In [34]:
products = pandas.read_csv("./Olist/olist_products_dataset.csv")
products

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


Здесь все кажется очевидным: в качестве первичного ключа будем использовать `product_id`, а `product_category_name` - внешний ключ на таблицу `translation`.

In [35]:
products_data_columns = products.columns[2:]
products_data_columns

Index(['product_name_lenght', 'product_description_lenght',
       'product_photos_qty', 'product_weight_g', 'product_length_cm',
       'product_height_cm', 'product_width_cm'],
      dtype='object')

In [36]:
products["product_id"] = products["product_id"].apply(hash_id)
products = products.merge(translation, on = "product_category_name", how = "left")
products = products[["product_id", "index", *products_data_columns]]
products

,product_id,index,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1865700292,6.0,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,1466070601,46.0,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,1980439124,5.0,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,-822471665,11.0,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,1023800763,7.0,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,-1845102617,4.0,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,-231384424,43.0,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,1563509476,3.0,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,1675541932,1.0,60.0,156.0,2.0,700.0,31.0,13.0,20.0


In [37]:
products.isna().sum(axis = 0) # Есть один объект с пропусками

product_id                      0
index                         623
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [38]:
# Кажется, что столбцы с вещественными значениями на самом деле целочисленные. Проверим это.
for column in products_data_columns:
    diff = products[column] - products[column].astype('Int64')
    print(f"{column}: ", (diff > 0).any())

# Действительно так, приведем их к целочисленному типу
for column in products_data_columns:
    products[column] = products[column].astype('Int64')
products

product_name_lenght:  False
product_description_lenght:  False
product_photos_qty:  False
product_weight_g:  False
product_length_cm:  False
product_height_cm:  False
product_width_cm:  False


,product_id,index,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1865700292,6.0,40,287,1,225,16,10,14
1,1466070601,46.0,44,276,1,1000,30,18,20
2,1980439124,5.0,46,250,1,154,18,9,15
3,-822471665,11.0,27,261,1,371,26,4,26
4,1023800763,7.0,37,402,4,625,20,17,13
...,...,...,...,...,...,...,...,...,...
32946,-1845102617,4.0,45,67,2,12300,40,40,40
32947,-231384424,43.0,41,971,1,1700,16,19,16
32948,1563509476,3.0,50,799,1,1400,27,7,27
32949,1675541932,1.0,60,156,2,700,31,13,20


In [39]:
duck.execute("""
CREATE TABLE olist.products (
    id INTEGER PRIMARY KEY,
    category_id INTEGER,
    name_length SMALLINT CHECK(name_length > 0),
    description_length SMALLINT CHECK(description_length > 0),
    photos_qty SMALLINT CHECK(photos_qty > 0),
    weight_g INTEGER CHECK(weight_g >= 0),
    length_cm INTEGER CHECK(length_cm > 0),
    height_cm INTEGER CHECK(height_cm > 0),
    width_cm INTEGER CHECK(width_cm > 0),

    FOREIGN KEY (category_id) REFERENCES olist.translation(id)
);
""")
duck.execute("INSERT INTO olist.products SELECT * FROM products")

### olist_order_items_dataset.csv

In [40]:
order_items = pandas.read_csv("./Olist/olist_order_items_dataset.csv")
order_items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72


In [41]:
order_items.nunique()

order_id               98666
order_item_id             21
product_id             32951
seller_id               3095
shipping_limit_date    93318
price                   5968
freight_value           6999
dtype: int64

Столбцы `product_id`, `seller_id` и `order_id`, очевидно, являются внешними ключами на таблицы `products`, `sellers` и `orders`. Тем не менее явного первичного ключа снова нет, но, по аналогии с данными платежей, кажется, что пара `(order_id, order_item_id)` должна подойти в качестве первичного ключа.

In [42]:
order_items["order_id"] = order_items["order_id"].apply(hash_id)
order_items["product_id"] = order_items["product_id"].apply(hash_id)
order_items["seller_id"] = order_items["seller_id"].apply(hash_id)
order_items.insert(0, "id", (order_items["order_id"] - order_items["order_item_id"] + 1).astype(int))
order_items = order_items.drop(columns = ['order_item_id'])
order_items

,id,order_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,1684711000,1684711000,-184576558,-1528358976,2017-09-19 09:45:35,58.90,13.29
1,-871234356,-871234356,1198843617,-1097508718,2017-05-03 11:05:13,239.90,19.93
2,-1591014014,-1591014014,1283045451,90442714,2018-01-18 14:48:30,199.00,17.87
3,946144102,946144102,-1243824609,87626403,2018-08-15 10:10:18,12.99,12.79
4,1613749990,1613749990,752593792,1373355143,2017-02-13 13:57:51,199.90,18.14
...,...,...,...,...,...,...,...
112645,1389450027,1389450027,854026777,-740000983,2018-05-02 04:11:01,299.99,43.41
112646,-1364205356,-1364205356,-1428648092,-2014570614,2018-07-20 04:31:48,350.00,36.53
112647,-78748535,-78748535,-540827908,487411460,2017-10-30 17:14:25,99.90,16.95
112648,-812117615,-812117615,107069564,-74425125,2017-08-21 00:04:32,55.99,8.72


In [43]:
duck.execute("""
CREATE TABLE olist.order_items (
    id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    seller_id INTEGER NOT NULL,
    shipping_limit_date DATETIME NOT NULL,
    price DOUBLE NOT NULL,
    freight_value DOUBLE NOT NULL,

    FOREIGN KEY (order_id) REFERENCES olist.orders(id),
    FOREIGN KEY (product_id) REFERENCES olist.products(id),
    FOREIGN KEY (seller_id) REFERENCES olist.sellers(id)
);
""")
# Наверняка мы захотим искать по order_id, product_id и seller_id, поэтому сразу подгоготовим индексы
duck.execute("CREATE INDEX order_items_order_id ON olist.order_items (order_id)")
duck.execute("CREATE INDEX order_items_product_id ON olist.order_items (product_id)")
duck.execute("CREATE INDEX order_items_seller_id ON olist.order_items (seller_id)")

duck.execute("INSERT INTO olist.order_items SELECT * FROM order_items")